# Customer Churn — Internship Practical
End-to-end Python + Pandas + EDA + Machine Learning workflow.

## 0. Imports and Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv("customer_churn.csv")
df.head()

# 1. Data Understanding
- Load dataset
- Check dimensions
- Understand columns and data types
- Identify target
- Check basic statistics

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nInfo:")
df.info()
print("\nStatistics:")
display(df.describe(include="all").T)

target = "Churn"
print("\nTarget:", target)

# 2. Data Cleaning
- Missing values
- Duplicate records
- Incorrect data types
- Handle issues

In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False))
print("Duplicates:", df.duplicated().sum())

# TotalCharges is conceptually numeric but may be stored as object
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("\nMissing values after conversion:")
display(df.isnull().sum().sort_values(ascending=False))

df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())
df = df.drop_duplicates().copy()
print("Final shape:", df.shape)

# 3. EDA
## 3.1 Target Variable

In [ ]:
print(df["Churn"].value_counts())
print("\nPercentage:")
print(df["Churn"].value_counts(normalize=True) * 100)

sns.countplot(data=df, x="Churn")
plt.title("Churn Distribution")
plt.show()

## 3.2 Numerical Variables

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

display(df[num_cols].describe().T)

for col in num_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(data=df, x=col, kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

for col in num_cols:
    plt.figure(figsize=(7,4))
    sns.boxplot(data=df, x="Churn", y=col)
    plt.title(f"{col} vs Churn")
    plt.show()

display(df.groupby("Churn")[num_cols].mean())
display(df.groupby("Churn")[num_cols].median())

## 3.3 Categorical Variables

In [ ]:
cat_cols = [
    "gender","SeniorCitizen","Partner","Dependents","PhoneService",
    "MultipleLines","InternetService","OnlineSecurity","OnlineBackup",
    "DeviceProtection","TechSupport","StreamingTV","StreamingMovies",
    "Contract","PaperlessBilling","PaymentMethod"
]

for col in cat_cols:
    print(f"\n{col}")
    print(df[col].value_counts())
    plt.figure(figsize=(8,4))
    sns.countplot(data=df, x=col, hue="Churn")
    plt.title(f"{col} vs Churn")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 3.4 Churn Rate by Category + Correlation

In [ ]:
for col in cat_cols:
    rate = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
    if "Yes" in rate.columns:
        print(f"Churn rate: {col}")
        display(rate.sort_values("Yes", ascending=False))

plt.figure(figsize=(7,5))
sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm")
plt.title("Numerical Correlation")
plt.show()

## 3.5 Outlier Check

In [ ]:
for col in num_cols:
    plt.figure(figsize=(7,3))
    sns.boxplot(x=df[col])
    plt.title(f"Outlier Check: {col}")
    plt.show()

# 4. Feature Preparation
- Remove identifier
- Encode target
- Separate X and y
- Train/test split
- One-hot encode categorical features
- Scale numerical features where appropriate

In [ ]:
df_model = df.drop(columns=["customerID"]).copy()

df_model["Churn"] = df_model["Churn"].map({"No": 0, "Yes": 1})

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = X.select_dtypes(include=["object"]).columns.tolist()
numerical_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

print("Categorical:", categorical_features)
print("Numerical/binary:", numerical_features)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical_features)
])

# 5. Machine Learning
Build and compare at least two classification models.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight="balanced"
    ))
])

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)
print("Models trained.")

# 6. Evaluation
Compare accuracy, precision, recall, F1-score and confusion matrix.

In [ ]:
def evaluate_model(name, model):
    y_pred = model.predict(X_test)

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }

    print("\n", name)
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["No Churn","Churn"],
                yticklabels=["No Churn","Churn"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    return metrics

results = [
    evaluate_model("Logistic Regression", logistic_model),
    evaluate_model("Random Forest", random_forest_model)
]

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df)

# 7. Final Interview Questions

Be ready to explain:
1. Why did you remove customerID?
2. Why was TotalCharges converted to numeric?
3. Why did you use one-hot encoding?
4. Why did you scale the numerical features?
5. Why use a pipeline?
6. Why split before fitting preprocessing?
7. Why use stratify=y?
8. Is the target imbalanced?
9. Why is accuracy not enough?
10. Which metric is important for churn and why?
11. Why did you choose Logistic Regression and Random Forest?
12. Does Random Forest need scaling?
13. What is data leakage?
14. What is overfitting and how would you reduce it?
15. Would you use SMOTE? Where would you apply it?
16. How would you improve the model?
17. How would you deploy it?
18. How would you monitor it after deployment?